# N110 · 换根DP与全节点答案

**目标：** 从一个根的统计量推导相邻根的答案。

**先修：** N109, N075, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 子树大小；两遍DFS；内外贡献；重根转移；距离和。

**配套讲解来源：** [同名逐章意见](../../comment/110_rerooting_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

树上有一类问题长这个样子："如果把**每个**节点分别当作根，统计某个量，再汇总（比如取个列表、数有多少个根达标）。"最笨的办法是对每个根都从头算一遍，n 个根就算 n 遍。本章要学的是"换根 DP"：只在根 0 上认真算一次，然后利用"相邻两个根的答案只差一点点"这个性质，把答案沿着树边一格格推出去。

两个代表问题：

1. **树上距离和**：对每个节点 i，求"以 i 为根时，所有节点到 i 的距离之和"，返回长度 n 的列表。（LeetCode 834）
2. **猜父节点计数**：给你一堆"方向猜测"（每个猜某条边的父子方向），问有多少个根能让你至少猜对 k 条。（LeetCode 2581）

先看一个具体到数字的例子（“运行示例”部分用的就是它）：树有 5 个点，边是 `[(0,1),(1,2),(1,3),(3,4)]`，形状是 0—1，1 又分叉出 2 和 3，3 带着孩子 4。

- 输入 `n=5, edges=[(0,1),(1,2),(1,3),(3,4)]`。
- 输出 `[8, 5, 8, 6, 9]`。
- 为什么 answer[1]=5？以 1 为根时各点距离是 0:1、1:0、2:1、3:1、4:2，加起来 1+0+1+1+2=5。1 在树的"中间"，所以它的距离和最小；4 是最偏的叶子，距离和最大（9）。第三节我们会看代码怎么从 answer[0]=8 一路推出这五个数。

再给一个小例子验证直觉：三点链 `0—1—2`，答案是 `[3,2,3]`（中间点最划算），这是“自动测试”部分的第一条断言。

## 2. 基础知识：先读懂这些词

- **换根 DP（rerooting）**：先在固定根上做一次完整的统计，然后"把根挪到隔壁点"，用一个 O(1) 的公式修正答案，依次把所有点的答案推出来。它把 O(n²) 的"每个根算一遍"降到 O(n)。
- **两遍扫描（自底向上 + 自顶向下）**：本章代码不写 DFS 递归，而是先 `_tree_order` 拿到一个"父在前、子在后"的 BFS 序 `order`。正着扫 `order` 是从根往下传信息；倒着扫 `reversed(order)` 是从叶往根收信息。两遍各干一件事，这就是"两遍 DFS"的迭代版。
- **子树大小 `size[u]`**：以根 0 为准，u 的子树里（含 u 自己）有多少个点。叶子是 1，父亲把自己的 size 加上孩子的 size，倒序一扫就有。
- **深度 `depth[u]`**：u 到根 0 的边数。所有深度之和就是"以 0 为根的距离和"（answer[0] 的值），正序一扫就有。
- **内外贡献**：把根从 u 挪到孩子 v 时，树上的点分成两伙——v 子树**内部**的 size[v] 个点和**外部**的 n−size[v] 个点。内部的每个点到新根近了一格（−1），外部的每个点远了一格（+1）。分"内/外"两伙算账，是换根公式的全部来源。
- **重根转移（换根公式）**：`answer[v] = answer[u] + n - 2*size[v]`。它就是上一条的两伙账合并：变化量 = −size[v] + (n−size[v]) = n − 2·size[v]。
- **边的方向翻转**：猜父节点问题里，把根从 p 挪到 u 时，树上**只有边 (p,u) 的方向翻了**（原来是 p→u，现在 u→p），其余所有边的父子方向都不变。所以得分只需要"扣掉旧方向的猜测、加上新方向的猜测"，O(1) 完成修正。

## 3. 思路推导：从小例子开始

### 主线：距离和，在 5 点树上手算

- **Step 1：选根 0，算深度。** 正着扫 order（`[0,1,2,3,4]`），每个点的深度 = 父亲深度 + 1：depth = [0,1,2,2,3]。
- **Step 2：answer[0] = 深度总和。** 0+1+2+2+3 = 8。
- **Step 3：倒着扫算子树大小。** 从叶子往根：size[4]=1；size[3]=1+size[4]=2；size[2]=1；size[1]=1+size[2]+size[3]=4；size[0]=1+4=5。
- **Step 4：换根公式沿 order 下推。** 对每个非根点 u（按 order 顺序，保证父亲的答案先算好）：`answer[u] = answer[父亲] + n − 2*size[u]`。
  - answer[1] = answer[0] + 5 − 2×4 = 8 − 3 = **5**。直觉核对：根从 0 挪到 1，1 的子树（4 个点）全近一格（−4），只剩 0 一个点远一格（+1），净变化 −3。
  - answer[2] = answer[1] + 5 − 2×1 = 5 + 3 = **8**。根从 1 挪到 2，只有 2 自己近一格，其余 4 个点（包括 1）各远一格，净 +3。
  - answer[3] = answer[1] + 5 − 2×2 = 5 + 1 = **6**。3 的子树 2 个点各近一格，3 个点各远一格。
  - answer[4] = answer[3] + 5 − 2×1 = 6 + 3 = **9**。
- **Step 5：汇总成“运行示例”部分那张表。**

| 根 | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| 距离总和 | 8 | 5 | 8 | 6 | 9 |

手工抽查 answer[4]=9：以 4 为根各点距离是 3,2,3,1,0，和为 9，对上了。

### 副线：猜父节点计数的小例子

沿用 5 点树。以 0 为根时的父子方向是 0→1、1→2、1→3、3→4。假设猜测集合是 `{(0,1),(1,2),(4,3)}`、k=2：

- 以 0 为根：对的方向有 (0,1)、(1,2) 两条，score[0]=2（(4,3) 是反的）。
- 把根挪到 1：边 (0,1) 方向翻转成 1→0。猜测里没有 (1,0)，所以扣掉原来对的 (0,1)（−1）；边 (0,1) 的新方向 (1,0) 不在猜测里（+0）。score[1] = 2 − 1 + 0 = 1。其余边方向没变，不用动。
- 依此把每个点都过一遍，最后数有多少个点的 score ≥ k。

把换根 DP 的套路总结成一句话：先老老实实在根 0 上把统计量算对，再找"把根挪过一条边"时统计量的 O(1) 修正公式，最后沿树序把修正传播到每个点。距离和的修正是 `n−2·size[v]`，方向得分的修正是"扣旧方向、加新方向"，形态不同、骨架相同。

## 4. 核心代码：sum_of_distances_in_tree

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def sum_of_distances_in_tree(n, edges):
    if not n:
        return []
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    size = [1] * n
    depth = [0] * n
    for u in order[1:]:
        depth[u] = depth[parent[u]] + 1
    for u in reversed(order[1:]):
        size[parent[u]] += size[u]
    answer = [0] * n
    answer[0] = sum(depth)
    for u in order[1:]:
        answer[u] = answer[parent[u]] + n - 2 * size[u]
    return answer
```

### 逐段读懂代码

### 辅助函数

```python
def _tree_order(adj, root=0):
    if not adj:
        return ([], [])
    parent = [-1] * len(adj)
    parent[root] = root
    order = [root]
    for u in order:
        for v in adj[u]:
            if v == parent[u]:
                continue
            if parent[v] != -1:
                raise ValueError('tree required')
            parent[v] = u
            order.append(v)
    if len(order) != len(adj):
        raise ValueError('connected tree required')
    return (parent, order)
```

从 root 出发做 BFS：`order` 是"父在前、子在后"的访问序列，`parent[v]` 记录每个点的父亲（根的父亲设成自己，方便统一判断"别走回头路"）。两个 `raise` 分别拦截"有环或重边"（邻居 v 已经有父亲却不是我的父亲）和"图不连通"（走完了还有点没访问到）。这两个守卫很重要：换根公式只对真正的树成立。

```python
def _adj_from_edges(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj
```

把边列表翻成邻接表：每条无向边在两头的列表里各登记一次。

### 1. `sum_of_distances_in_tree`

```python
def sum_of_distances_in_tree(n, edges):
    if not n:
        return []
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    size = [1] * n
    depth = [0] * n
    for u in order[1:]:
        depth[u] = depth[parent[u]] + 1
    for u in reversed(order[1:]):
        size[parent[u]] += size[u]
    answer = [0] * n
    answer[0] = sum(depth)
    for u in order[1:]:
        answer[u] = answer[parent[u]] + n - 2 * size[u]
    return answer
```

逐行看：

- `if not n: return []`：空图直接返回空列表，挡边界。
- `size=[1]*n`：每个点初始算上自己共 1 个。
- 第一个循环正序扫 `order[1:]`（跳过根 0）：父亲深度已算好，`depth[u]=depth[parent[u]]+1`。
- 第二个循环倒序扫：孩子的 size 先累加完，再把自己整份交给父亲 `size[parent[u]]+=size[u]`。这就是"自底向上"的那一遍。
- `answer[0]=sum(depth)`：以 0 为根的距离和就是深度总和，第一遍的成果直接复用。
- 第三个循环再正序扫：此时每个点的 size 和父亲的 answer 都就绪了，套换根公式 `answer[u]=answer[parent[u]]+n-2*size[u]`（第三节 Step 4 手算过）。这是"自顶向下"的那一遍。

三遍线性扫，没有递归、没有重复统计。

### 2. `root_count`

```python
def root_count(edges, guesses, k):
    n = len(edges) + 1
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    guesses = set(map(tuple, guesses))
    score = [0] * n
    score[0] = sum(((parent[u], u) in guesses for u in order[1:]))
    for u in order[1:]:
        p = parent[u]
        score[u] = score[p] - ((p, u) in guesses) + ((u, p) in guesses)
    return sum((x >= k for x in score))
```

- `n=len(edges)+1`：树的点数比边数多一，从输入就地推出。
- `guesses=set(map(tuple,guesses))`：把猜测列表转成集合，后面每条边的两次查询都是 O(1) 平均。
- `score[0]=sum((parent[u],u) in guesses for u in order[1:])`：以 0 为根时，每条边的真实方向是 (父,子)，即 `(parent[u],u)`；数它出现在猜测集合里的次数，就是根 0 的得分。`in` 返回布尔值，sum 把 True 当 1 加。
- 转移那一行是换根公式的"方向版"：把根从 p 挪到 u，只有边 (p,u) 翻转。旧方向 (p,u) 如果被猜中了，现在它错了，扣掉（`-((p,u) in guesses)`）；新方向 (u,p) 如果也被猜了，现在它对了，加上（`+((u,p) in guesses)`）。其他边一概不动。
- `return sum(x>=k for x in score)`：数有多少个根的得分达到 k。

## 5. 分段实现：按顺序运行

**本章接口：** `sum_of_distances_in_tree(n, edges)`、`root_count(edges, guesses, k)`

### _tree_order

In [1]:
def _tree_order(adj, root=0):
    if not adj:
        return ([], [])
    parent = [-1] * len(adj)
    parent[root] = root
    order = [root]
    for u in order:
        for v in adj[u]:
            if v == parent[u]:
                continue
            if parent[v] != -1:
                raise ValueError('tree required')
            parent[v] = u
            order.append(v)
    if len(order) != len(adj):
        raise ValueError('connected tree required')
    return (parent, order)

### _adj_from_edges

In [2]:
def _adj_from_edges(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj

### sum_of_distances_in_tree

In [3]:
def sum_of_distances_in_tree(n, edges):
    if not n:
        return []
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    size = [1] * n
    depth = [0] * n
    for u in order[1:]:
        depth[u] = depth[parent[u]] + 1
    for u in reversed(order[1:]):
        size[parent[u]] += size[u]
    answer = [0] * n
    answer[0] = sum(depth)
    for u in order[1:]:
        answer[u] = answer[parent[u]] + n - 2 * size[u]
    return answer

### root_count

In [4]:
def root_count(edges, guesses, k):
    n = len(edges) + 1
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    guesses = set(map(tuple, guesses))
    score = [0] * n
    score[0] = sum(((parent[u], u) in guesses for u in order[1:]))
    for u in order[1:]:
        p = parent[u]
        score[u] = score[p] - ((p, u) in guesses) + ((u, p) in guesses)
    return sum((x >= k for x in score))

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(1,2),(1,3),(3,4)]; answer=sum_of_distances_in_tree(5,edges)
show_table(['根','距离总和'],list(enumerate(answer)))

根,距离总和
0,8
1,5
2,8
3,6
4,9


## 7. 正确性、适用条件与复杂度

删除边(u,v)将树分成互补两部分，任意点到两个端点的路径必须经过其中一个端点，故距离变化恰为±1。先算根0，再沿树边传播的归纳覆盖全部根。方向得分只扣除旧方向并加入新方向。

**代价：** O(n+|guesses|)平均时间和O(n+|guesses|)空间。guesses按集合解释；编号0..n−1，无向边构成树。

### 展开理解

**为什么是对的？** 先说距离和公式为什么恰好是 ±1 的账。把根从 u 挪到相邻的孩子 v，唯一改变的是"你在这条边 (u,v) 的哪一侧"这个事实：从任何一个点走到根，路径必须横穿这条边，位置从 v 侧换成 u 侧，路径长度恰好变化 1——v 子树内部的 size[v] 个点（含 v）原来要多跨这条边，现在不用了，各减 1；外面的 n−size[v] 个点原来不用跨，现在要跨，各加 1。所以总变化 = −size[v] + (n−size[v]) = n−2·size[v]，不多不少。接着我们论证所有根的答案都会被算出来：answer[0] 由深度和直接给出；只要某个根 u 的答案对了，它每个孩子的答案就由公式推出；从根 0 沿着树边一步步传，树是连通的，每个点都会被传到。这就是"先算根 0，再沿树边传播覆盖全部根"的完整论证。猜父节点那边同理：挪根只翻转被跨越的那一条边，所以得分变化就是"旧方向那条猜测从对变错、新方向那条从错变对"，其余猜测的对错状态纹丝不动。

**复杂度**：`sum_of_distances_in_tree` 三遍线性扫描，O(n) 时间、O(n) 空间。`root_count` 也是 O(n) 次转移，外加把猜测装进集合的 O(|guesses|)，所以是 O(n+|guesses|) 时间和空间（集合查询是平均 O(1)）。拿具体数字感受：n = 10⁵ 个点时，暴力"每个根各算一遍"是约 10¹⁰ 次操作（上百秒，直接超时），而换根只要三十万级操作，一瞬间出结果——这就是这一章存在的意义。前提：输入编号是 0..n−1、边构成一棵连通无向树，`_tree_order` 的两个 raise 会替你把关。

## 8. 单元测试：每个用例在检查什么

- `assert sum_of_distances_in_tree(3,[(0,1),(1,2)])==[3,2,3]`：正常值测试，三点链。中间点的距离和 2 最小、两端各是 3，检查深度求和与换根公式在最小非平凡树上都对。
- `assert sum_of_distances_in_tree(1,[])==[0]`：边界测试，单点树没有边，唯一可能的根距离和是 0；它同时覆盖 `if not n` 没挡到的"n=1 但非空"情形，考验 `order[1:]` 为空时各循环是否安然跳过。
- 后半部分是随机对拍：`rng=Random(110)` 固定种子可复现；对 n=1..15 各造一棵随机树（每个新点 v 随机挂在编号更小的点上）。裁判是对每个根 r 做一次真正的 BFS：`d` 数组给出各点距离，`reference` 收集距离和；同时把 BFS 实际走过边的方向 `(u,v)` 收进集合 `pairs`，与猜测集合做交集就是该根的得分 `scores`。断言本章函数与这个"每个根真算一遍"的参照完全一致，并对所有 k 检查 `root_count` 等于"得分 ≥ k 的根的个数"。注意参照实现本身就是 O(n²) 的暴力——它慢，但显然忠实于定义，正适合当裁判。

In [6]:
assert sum_of_distances_in_tree(3, [(0, 1), (1, 2)]) == [3, 2, 3]

assert sum_of_distances_in_tree(1, []) == [0]

from random import Random

from collections import deque

rng = Random(110)

for n in range(1, 16):
    edges = [(rng.randrange(v), v) for v in range(1, n)]
    adj = _adj_from_edges(n, edges)
    guesses = [(v, u) if rng.randrange(2) else (u, v) for u, v in edges]
    reference = []
    scores = []
    for root in range(n):
        d = [-1] * n
        d[root] = 0
        q = deque([root])
        pairs = set()
        while q:
            u = q.popleft()
            for v in adj[u]:
                if d[v] == -1:
                    d[v] = d[u] + 1
                    q.append(v)
                    pairs.add((u, v))
        reference.append(sum(d))
        scores.append(len(pairs & set(guesses)))
    assert sum_of_distances_in_tree(n, edges) == reference
    for k in range(n + 1):
        assert root_count(edges, guesses, k) == sum((x >= k for x in scores))

print('N110: 所有本章断言通过')

N110: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导ans[v]=ans[u]+n-2*size[v]。

**练习 2：** 构造链和星形的闭式答案。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（推导 ans[v]=ans[u]+n−2·size[v]）**：提示——从"距离和怎么变"入手，不要背公式。把点分成两伙：v 子树里 size[v] 个、外面 n−size[v] 个。挪根跨越边 (u,v) 后，里伙每人近 1、外伙每人远 1，总变化 −size[v]+(n−size[v])。用“运行示例”部分那棵 5 点树手算一遍（第三节 Step 4 已经示范），再用 `sum_of_distances_in_tree` 或“自动测试”部分的 BFS 暴力对照。边界别忘了：size[v] 恰好等于 n 时（v 是唯一的"孩子"且树退化成两点）公式退化为 answer[v]=answer[u]−n 的情况也要说得通。
- **练习 2（链和星形的闭式答案）**：提示——"闭式"就是不跑代码、用公式直接写出答案。链 0—1—2—…−(n−1)：以位置 i 为根的距离和是 Σ|i−j|，拆成两段等差数列求和，得到 i(i+1)/2 + (n−1−i)(n−i)/2；星形（中心连着 n−1 个叶子）：中心为根答案是 n−1，任何叶子为根是 (n−2)×2 + 1 = 2n−3（其他叶子都远 2，中心远 1）。先手算 n=5 的链（答案 [10,7,6,7,10]）和 5 点星（[4,7,7,7,7]）验证你的公式，再调本章接口对答案。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def _tree_order(adj, root=0):
    if not adj:
        return ([], [])
    parent = [-1] * len(adj)
    parent[root] = root
    order = [root]
    for u in order:
        for v in adj[u]:
            if v == parent[u]:
                continue
            if parent[v] != -1:
                raise ValueError('tree required')
            parent[v] = u
            order.append(v)
    if len(order) != len(adj):
        raise ValueError('connected tree required')
    return (parent, order)

def _adj_from_edges(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj

def sum_of_distances_in_tree(n, edges):
    if not n:
        return []
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    size = [1] * n
    depth = [0] * n
    for u in order[1:]:
        depth[u] = depth[parent[u]] + 1
    for u in reversed(order[1:]):
        size[parent[u]] += size[u]
    answer = [0] * n
    answer[0] = sum(depth)
    for u in order[1:]:
        answer[u] = answer[parent[u]] + n - 2 * size[u]
    return answer

def root_count(edges, guesses, k):
    n = len(edges) + 1
    adj = _adj_from_edges(n, edges)
    parent, order = _tree_order(adj)
    guesses = set(map(tuple, guesses))
    score = [0] * n
    score[0] = sum(((parent[u], u) in guesses for u in order[1:]))
    for u in order[1:]:
        p = parent[u]
        score[u] = score[p] - ((p, u) in guesses) + ((u, p) in guesses)
    return sum((x >= k for x in score))

# 示例与回归测试
assert sum_of_distances_in_tree(3, [(0, 1), (1, 2)]) == [3, 2, 3]

assert sum_of_distances_in_tree(1, []) == [0]

from random import Random

from collections import deque

rng = Random(110)

for n in range(1, 16):
    edges = [(rng.randrange(v), v) for v in range(1, n)]
    adj = _adj_from_edges(n, edges)
    guesses = [(v, u) if rng.randrange(2) else (u, v) for u, v in edges]
    reference = []
    scores = []
    for root in range(n):
        d = [-1] * n
        d[root] = 0
        q = deque([root])
        pairs = set()
        while q:
            u = q.popleft()
            for v in adj[u]:
                if d[v] == -1:
                    d[v] = d[u] + 1
                    q.append(v)
                    pairs.add((u, v))
        reference.append(sum(d))
        scores.append(len(pairs & set(guesses)))
    assert sum_of_distances_in_tree(n, edges) == reference
    for k in range(n + 1):
        assert root_count(edges, guesses, k) == sum((x >= k for x in scores))

print('N110: 所有本章断言通过')

N110: 所有本章断言通过


## 11. 代表题与迁移

- **834. Sum of Distances in Tree**：练"深度求和打底 + size 换根公式下推"的标准两遍扫描，本章 `sum_of_distances_in_tree` 即其解。
- **2581. Count Number of Possible Root Nodes**：练"换根不一定要算距离，任何随根变化的量只要能 O(1) 修正就能换根"——这里是边方向翻转的得分修正，对应 `root_count`。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。